# Introduction to Machine Learning – Lab 2 (Part 6)
## Hyperparameter Tuning & Bayesian Optimization with Optuna

### Objective
Building on our baseline analysis and data preparation insights, our goal in this section is to eliminate model overfitting (high variance) and push our **Macro-F1 score** as high as possible:
1. **Manual Regularization:** Testing targeted adjustments to tree depth, learning rate, and subsampling.
2. **Bayesian Optimization (`Optuna` + `TPESampler`):** Automatically exploring the hyperparameter space using 5-Fold Stratified Cross-Validation.
3. **SQLite Persistence:** Saving intermediate trial steps to prevent data loss.
4. **Optimization Visualizations:** Analyzing parameter importance and history plots.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path(".").resolve()))
sys.path.append(str(Path("notebooks").resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

from preprocessing import CLASS_NAMES, build_pipeline, prepare_data

# Suppress excessive Optuna logging output during optimization
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Load baseline dataset
DATA_PATH = Path("../data/AmesHousing.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/AmesHousing.csv")

X, y = prepare_data(
    DATA_PATH,
    drop_extreme=False,
    drop_near_constant=False,
)

print(f"Dataset shape: X = {X.shape}, y = {y.shape}")

Dataset shape: X = (2930, 79), y = (2930,)


### 1. Manual Hyperparameter Exploration

In Part 4, we saw that our baseline model achieved near $100\%$ accuracy on the training set but lagged on validation ($\sim 75\%$), indicating high variance. 

Let's test a manually regularized model by:
- Constraining tree depth (`max_depth=4` instead of default deeper trees).
- Lowering the learning rate (`learning_rate=0.05`).
- Adding row and column subsampling (`subsample=0.8`, `colsample_bytree=0.7`) to break feature correlation.
- Adding L2 regularization (`reg_lambda=1.0`).

In [3]:
from evaluation import make_splits, evaluate

# Regularized configuration
regularized_pipe = build_pipeline(
    strategy="onehot",
    min_frequency=5,
    enable_categorical=False,
    tree_method="hist",
    max_depth=4,
    learning_rate=0.05,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_lambda=1.0,
)

splits = make_splits(y)
reg_results = evaluate(regularized_pipe, X, y, splits)

cv5_reg = reg_results[reg_results["Setup"] == "CV k=5, stratified"]
print("--- Regularized Model Performance (5-Fold Stratified CV) ---")
print(f"Test Accuracy: {cv5_reg['Test Accuracy'].mean():.4f} ± {cv5_reg['Test Accuracy'].std():.4f}")
print(f"Test Macro-F1: {cv5_reg['Test Macro-F1'].mean():.4f} ± {cv5_reg['Test Macro-F1'].std():.4f}")

--- Regularized Model Performance (5-Fold Stratified CV) ---
Test Accuracy: 0.7532 ± 0.0163
Test Macro-F1: 0.7322 ± 0.0131


### 2. Bayesian Optimization with Optuna (`TPESampler`)

We now use Optuna with the **Tree-structured Parzen Estimator (`TPESampler`)** to search for the optimal hyperparameter combination. We save trials to an SQLite database (`optuna_ames_housing.db`) so progress is permanently cached[cite: 2].

In [4]:
import joblib

# Define the Optuna objective function optimizing 5-Fold Stratified CV Macro-F1
def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }

    pipeline = build_pipeline(
        strategy="onehot",
        min_frequency=5,
        enable_categorical=False,
        tree_method="hist",
        random_state=42,
        **params
    )

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_f1s = []

    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

        pipeline.fit(X_tr, y_tr)
        preds = pipeline.predict(X_va)
        fold_f1s.append(f1_score(y_va, preds, average="macro"))

    return np.mean(fold_f1s)

# Create SQLite storage for intermediate steps
db_path = "sqlite:///optuna_ames_housing.db"
study_name = "ames_xgboost_macro_f1"

study = optuna.create_study(
    study_name=study_name,
    storage=db_path,
    load_if_exists=True,
    sampler=TPESampler(seed=42),
    direction="maximize",
)

# Run optimization (e.g., 30 trials; increase to 50+ if you want a deeper search)
print("Starting Optuna Bayesian Optimization (this may take a couple of minutes)...")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("\n--- Optimization Complete ---")
print(f"Best Trial Macro-F1: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Starting Optuna Bayesian Optimization (this may take a couple of minutes)...


  0%|          | 0/30 [00:00<?, ?it/s]


--- Optimization Complete ---
Best Trial Macro-F1: 0.7487
Best Hyperparameters:
  max_depth: 3
  learning_rate: 0.17160445029754812
  n_estimators: 400
  subsample: 0.9233589392465844
  colsample_bytree: 0.6523068845866853
  min_child_weight: 1
  reg_alpha: 0.5456725485601477
  reg_lambda: 0.057624872164786026


### 3. Visualizing the Optimization Process

Optuna provides built-in visualization tools to inspect the search history and evaluate which hyperparameters had the biggest impact on our Macro-F1 score[cite: 2].

In [5]:
from optuna.visualization import plot_optimization_history, plot_param_importances

# 1. Optimization History (Macro-F1 progress across trials)
fig_hist = plot_optimization_history(study)
fig_hist.update_layout(title="Optuna Optimization History: Macro-F1 per Trial")
fig_hist.show()

# 2. Hyperparameter Importances (Which parameter mattered most?)
fig_imp = plot_param_importances(study)
fig_imp.update_layout(title="Hyperparameter Importance for Maximizing Macro-F1")
fig_imp.show()

### Summary of Findings

1. **Baseline vs. Regularized vs. Optimized:** Regularization successfully reduced overfitting by restricting unnecessary tree growth, while Optuna fine-tuned trade-offs between learning rate, subsampling, and regularization penalties.
2. **Class Imbalance Strategy:** Handling the minority classes (`≥ 400k`) via precision-recall threshold tuning and stratified splitting ensured stable evaluations.